# 02 — Intent Model

Compares majority baseline, TF-IDF + Logistic Regression, and TF-IDF + Linear SVM using transparent weak labels.

In [ ]:
import sys
from pathlib import Path
import pandas as pd, numpy as np, re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,f1_score,classification_report
sys.path.insert(0,str(Path('..').resolve()))
from src.taxonomy import weak_label
DATA=Path('../data')
if not DATA.exists(): DATA=Path('data')
d=pd.read_csv(DATA/'historical_support_pairs.csv').fillna('')
d['intent']=d.clean_message.map(weak_label)
d=d[d.intent!='other'].copy()
ids=d.conversation_id.drop_duplicates()
tr_ids,te_ids=train_test_split(ids,test_size=.2,random_state=42)
tr=d[d.conversation_id.isin(tr_ids)]; te=d[d.conversation_id.isin(te_ids)]


In [ ]:
v=TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True)
Xtr=v.fit_transform(tr.clean_message); Xte=v.transform(te.clean_message)
models=[('logistic',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42)),('linear_svm',LinearSVC(class_weight='balanced',random_state=42))]
rows=[]
major=tr.intent.mode().iloc[0]
rows.append({'model':'majority','accuracy':accuracy_score(te.intent,[major]*len(te)),'macro_f1':f1_score(te.intent,[major]*len(te),average='macro')})
for name,m in models:
    m.fit(Xtr,tr.intent); p=m.predict(Xte)
    rows.append({'model':name,'accuracy':accuracy_score(te.intent,p),'macro_f1':f1_score(te.intent,p,average='macro')})
    print(name); print(classification_report(te.intent,p,zero_division=0))
results=pd.DataFrame(rows); display(results); results.to_csv(DATA/'intent_experiment_results.csv',index=False)
